In [ ]:
from google.colab import files

uploaded = files.upload()


Saving dataset.zip to dataset.zip


In [ ]:
import zipfile

with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
import os

for folder in os.listdir("dataset"):
    count = len(os.listdir(f"dataset/{folder}"))
    print(f"{folder}: {count} images")

D: 100 images
B: 100 images
E: 100 images
C: 100 images
A: 100 images


In [ ]:
import os
import cv2
import numpy as np

X = []
y = []

labels = {
    'A': 0,
    'B': 1,
    'C': 2,
    'D': 3,
    'E': 4
}

for folder in os.listdir("dataset"):

    folder_path = os.path.join("dataset", folder)

    for image_name in os.listdir(folder_path):

        image_path = os.path.join(
            folder_path,
            image_name
        )

        img = cv2.imread(image_path)

        img = cv2.resize(img, (64, 64))

        X.append(img)

        y.append(labels[folder])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (500, 64, 64, 3)
y shape: (500,)


In [ ]:
X = X.astype("float32") / 255.0

print("Min Value:", X.min())
print("Max Value:", X.max())

Min Value: 0.015686275
Max Value: 0.9019608


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (400, 64, 64, 3)
X_test : (100, 64, 64, 3)
y_train: (400,)
y_test : (100,)


In [ ]:
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(y_train, num_classes=5)
y_test = to_categorical(y_test, num_classes=5)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("\nFirst Label:")
print(y_train[0])

y_train shape: (400, 5)
y_test shape : (100, 5)

First Label:
[0. 0. 0. 1. 0.]


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense
)

model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(64,64,3)
    ),

    MaxPooling2D((2,2)),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D((2,2)),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),

    Dense(
        5,
        activation='softmax'
    )
])

model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,625,797 (6.20 MB)

 Trainable params: 1,625,797 (6.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model Compiled Successfully")

Model Compiled Successfully


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.8125 - loss: 0.5912 - val_accuracy: 0.8300 - val_loss: 0.5653
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8125 - loss: 0.5370 - val_accuracy: 0.7800 - val_loss: 0.5665
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.8350 - loss: 0.4924 - val_accuracy: 0.8400 - val_loss: 0.4824
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.8775 - loss: 0.4103 - val_accuracy: 0.8200 - val_loss: 0.4632
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8575 - loss: 0.3753 - val_accuracy: 0.8500 - val_loss: 0.3859
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8950 - loss: 0.3505 - val_accuracy: 0.8800 - val_loss: 0.3239
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8850 - loss: 0.2992 - val_accuracy: 0.8800 - val_loss: 0.3034
Epoch 8/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9250 - loss: 0.2542 - val_accuracy: 0.9200 - v

In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", accuracy)
print("Test Loss:", loss)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9700 - loss: 0.1064 
Test Accuracy: 0.9700000286102295
Test Loss: 0.10638427734375


In [ ]:
model.save("sign_language_model.keras")

In [ ]:
from google.colab import files

files.download("sign_language_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>